# SentimentForge — BiLSTM & GRU Experiments

This notebook explores advanced recurrent neural network
architectures for IMDb movie review sentiment classification.

Objectives:
- Understand bidirectional LSTM
- Implement a BiLSTM model
- Understand GRU architecture
- Implement a GRU model
- Train and evaluate both models
- Compare RNN, LSTM, BiLSTM and GRU

In [ ]:
import os
import json
import pickle
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/SentimentForge"

MODEL_DIR = os.path.join(PROJECT_DIR, "models")
DATA_DIR = os.path.join(PROJECT_DIR, "data")
REPORT_DIR = os.path.join(PROJECT_DIR, "reports")

DATASET_PATH = os.path.join(
    DATA_DIR, "raw", "IMDB Dataset.csv"
)

TOKENIZER_PATH = os.path.join(MODEL_DIR, "tokenizer.pkl")
CONFIG_PATH = os.path.join(MODEL_DIR, "preprocessing_config.json")

LSTM_MODEL_PATH = os.path.join(MODEL_DIR, "lstm_model.pth")

print("Project paths configured!")

Project paths configured!


In [ ]:
# Load IMDb dataset
df = pd.read_csv(DATASET_PATH)

# Load tokenizer
with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)

# Load preprocessing configuration
with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

print("Dataset shape:", df.shape)
print("Vocabulary size:", len(tokenizer.word_index))
print("Maximum vocabulary:", config["max_words"])
print("Maximum sequence length:", config["max_length"])

print("\nLabel distribution:")
print(df["sentiment"].value_counts())

Dataset shape: (50000, 2)
Vocabulary size: 107696
Maximum vocabulary: 30000
Maximum sequence length: 300

Label distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [ ]:
import re
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split

def clean_review(text):
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_review"] = df["review"].apply(clean_review)

label_map = {
    "negative": 0,
    "positive": 1
}

df["label"] = df["sentiment"].map(label_map)

X = df["clean_review"]
y = df["label"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Train: 40000
Validation: 5000
Test: 5000


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_WORDS = config["max_words"]
MAX_LEN = config["max_length"]

# Convert text into integer sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print("Training:", X_train_pad.shape)
print("Validation:", X_val_pad.shape)
print("Testing:", X_test_pad.shape)

Training: (40000, 300)
Validation: (5000, 300)
Testing: (5000, 300)


In [ ]:
# Convert to tensors
X_train_tensor = torch.tensor(X_train_pad, dtype=torch.long)
X_val_tensor = torch.tensor(X_val_pad, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_pad, dtype=torch.long)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# DataLoaders
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 625
Validation batches: 79
Testing batches: 79


In [ ]:
class BiLSTMModel(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(BiLSTMModel, self).__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):

        embedded = self.embedding(x)

        output, (hidden, cell) = self.lstm(embedded)

        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]

        combined_hidden = torch.cat(
            (forward_hidden, backward_hidden), dim=1
        )

        prediction = self.fc(combined_hidden)

        return prediction.squeeze(1)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [ ]:
VOCAB_SIZE = config["max_words"]
EMBEDDING_DIM = 128
HIDDEN_DIM = 128

bilstm_model = BiLSTMModel(
    VOCAB_SIZE,
    EMBEDDING_DIM,
    HIDDEN_DIM
).to(device)

total_params = sum(
    p.numel() for p in bilstm_model.parameters()
    if p.requires_grad
)

print(bilstm_model)
print("\nTrainable parameters:", total_params)

BiLSTMModel(
  (embedding): Embedding(30000, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)

Trainable parameters: 4104449


In [ ]:
sample_X, sample_y = next(iter(train_loader))

sample_X = sample_X.to(device)

bilstm_model.eval()

with torch.no_grad():
    bilstm_output = bilstm_model(sample_X)

print("Input shape:", sample_X.shape)
print("Output shape:", bilstm_output.shape)
print("First 5 outputs:", bilstm_output[:5])

Input shape: torch.Size([64, 300])
Output shape: torch.Size([64])
First 5 outputs: tensor([-0.0080, -0.0301,  0.0656, -0.0184,  0.0296], device='cuda:0')


In [ ]:
bilstm_criterion = nn.BCEWithLogitsLoss()

bilstm_optimizer = torch.optim.Adam(
    bilstm_model.parameters(),
    lr=0.001
)

print("Loss function:", bilstm_criterion)
print("Optimizer:", bilstm_optimizer)

Loss function: BCEWithLogitsLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for inputs, labels in loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        total_loss += loss.item()

        # Convert logits into binary predictions
        probabilities = torch.sigmoid(outputs)
        predictions = (probabilities >= 0.5).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def evaluate_model(model, loader, criterion, device):

    model.eval()

    total_loss = 0
    all_labels = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():

        for inputs, labels in loader:

            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            probabilities = torch.sigmoid(outputs)
            predictions = (probabilities >= 0.5).float()

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())

    avg_loss = total_loss / len(loader)

    accuracy = accuracy_score(all_labels, all_predictions)
    precision = precision_score(all_labels, all_predictions)
    recall = recall_score(all_labels, all_predictions)
    f1 = f1_score(all_labels, all_predictions)
    roc_auc = roc_auc_score(all_labels, all_probabilities)

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc
    }

In [ ]:
EPOCHS = 5

bilstm_history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(
        bilstm_model,
        train_loader,
        bilstm_criterion,
        bilstm_optimizer,
        device
    )

    val_metrics = evaluate_model(
        bilstm_model,
        val_loader,
        bilstm_criterion,
        device
    )

    bilstm_history["train_loss"].append(train_loss)
    bilstm_history["train_accuracy"].append(train_acc)
    bilstm_history["val_loss"].append(val_metrics["loss"])
    bilstm_history["val_accuracy"].append(val_metrics["accuracy"])

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.4f}")
    print("-" * 50)

Epoch 1/5
Train Loss: 0.6579 | Train Acc: 0.6095
Val Loss:   0.5976 | Val Acc: 0.6864
--------------------------------------------------
Epoch 2/5
Train Loss: 0.5682 | Train Acc: 0.7083
Val Loss:   0.5245 | Val Acc: 0.7566
--------------------------------------------------
Epoch 3/5
Train Loss: 0.4714 | Train Acc: 0.7813
Val Loss:   0.4374 | Val Acc: 0.8134
--------------------------------------------------
Epoch 4/5
Train Loss: 0.3467 | Train Acc: 0.8557
Val Loss:   0.3853 | Val Acc: 0.8346
--------------------------------------------------
Epoch 5/5
Train Loss: 0.2599 | Train Acc: 0.8988
Val Loss:   0.3755 | Val Acc: 0.8442
--------------------------------------------------


In [ ]:
bilstm_test_metrics = evaluate_model(
    bilstm_model,
    test_loader,
    bilstm_criterion,
    device
)

print("BiLSTM Test Results")
print("-" * 35)

for metric, value in bilstm_test_metrics.items():
    print(f"{metric.capitalize():12s}: {value:.4f}")

BiLSTM Test Results
-----------------------------------
Loss        : 0.3795
Accuracy    : 0.8464
Precision   : 0.8876
Recall      : 0.7932
F1          : 0.8378
Roc_auc     : 0.9230


In [ ]:
class GRUModel(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim):

        super(GRUModel, self).__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):

        embedded = self.embedding(x)

        output, hidden = self.gru(embedded)

        last_hidden = hidden[-1]

        prediction = self.fc(last_hidden)

        return prediction.squeeze(1)

In [ ]:
gru_model = GRUModel(
    VOCAB_SIZE,
    EMBEDDING_DIM,
    HIDDEN_DIM
).to(device)

total_params = sum(
    p.numel()
    for p in gru_model.parameters()
    if p.requires_grad
)

print(gru_model)
print("\nTrainable parameters:", total_params)

GRUModel(
  (embedding): Embedding(30000, 128, padding_idx=0)
  (gru): GRU(128, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

Trainable parameters: 3939201


In [ ]:
sample_X, sample_y = next(iter(train_loader))

sample_X = sample_X.to(device)

gru_model.eval()

with torch.no_grad():
    gru_output = gru_model(sample_X)

print("Input shape:", sample_X.shape)
print("Output shape:", gru_output.shape)
print("First 5 outputs:", gru_output[:5])

Input shape: torch.Size([64, 300])
Output shape: torch.Size([64])
First 5 outputs: tensor([0.0935, 0.0935, 0.0935, 0.3378, 0.0935], device='cuda:0')


In [ ]:
gru_criterion = nn.BCEWithLogitsLoss()

gru_optimizer = torch.optim.Adam(
    gru_model.parameters(),
    lr=0.001
)

print("GRU loss:", gru_criterion)
print("GRU optimizer:", gru_optimizer)

GRU loss: BCEWithLogitsLoss()
GRU optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [ ]:
EPOCHS = 5

gru_history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(
        gru_model,
        train_loader,
        gru_criterion,
        gru_optimizer,
        device
    )

    val_metrics = evaluate_model(
        gru_model,
        val_loader,
        gru_criterion,
        device
    )

    gru_history["train_loss"].append(train_loss)
    gru_history["train_accuracy"].append(train_acc)
    gru_history["val_loss"].append(val_metrics["loss"])
    gru_history["val_accuracy"].append(val_metrics["accuracy"])

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.4f}")
    print("-" * 50)

Epoch 1/5
Train Loss: 0.6932 | Train Acc: 0.5071
Val Loss:   0.6819 | Val Acc: 0.5808
--------------------------------------------------
Epoch 2/5
Train Loss: 0.6604 | Train Acc: 0.5898
Val Loss:   0.4684 | Val Acc: 0.7904
--------------------------------------------------
Epoch 3/5
Train Loss: 0.3192 | Train Acc: 0.8627
Val Loss:   0.2782 | Val Acc: 0.8846
--------------------------------------------------
Epoch 4/5
Train Loss: 0.1907 | Train Acc: 0.9273
Val Loss:   0.2592 | Val Acc: 0.8934
--------------------------------------------------
Epoch 5/5
Train Loss: 0.1191 | Train Acc: 0.9597
Val Loss:   0.2957 | Val Acc: 0.8934
--------------------------------------------------


In [ ]:
gru_test_metrics = evaluate_model(
    gru_model,
    test_loader,
    gru_criterion,
    device
)

print("GRU Test Results")
print("-" * 35)

for metric, value in gru_test_metrics.items():
    print(f"{metric.capitalize():12s}: {value:.4f}")

GRU Test Results
-----------------------------------
Loss        : 0.3007
Accuracy    : 0.8884
Precision   : 0.8903
Recall      : 0.8860
F1          : 0.8881
Roc_auc     : 0.9571


In [ ]:
GRU_LAST_PATH = os.path.join(
    MODEL_DIR,
    "gru_epoch5_last.pth"
)

torch.save({
    "model_state_dict": gru_model.state_dict(),
    "vocab_size": VOCAB_SIZE,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
    "max_length": MAX_LEN,
    "epoch": 5,
    "test_metrics": gru_test_metrics
}, GRU_LAST_PATH)

print("Saved:", GRU_LAST_PATH)

Saved: /content/drive/MyDrive/SentimentForge/models/gru_epoch5_last.pth


In [ ]:
BILSTM_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "bilstm_model.pth"
)

torch.save({
    "model_state_dict": bilstm_model.state_dict(),
    "vocab_size": VOCAB_SIZE,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
    "max_length": MAX_LEN,
    "test_metrics": bilstm_test_metrics
}, BILSTM_MODEL_PATH)

print("Saved:", BILSTM_MODEL_PATH)

Saved: /content/drive/MyDrive/SentimentForge/models/bilstm_model.pth


In [ ]:
import json

# BiLSTM experiment record
bilstm_experiment = {
    "model": "BiLSTM",
    "epochs": 5,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
    "trainable_parameters": sum(
        p.numel() for p in bilstm_model.parameters()
        if p.requires_grad
    ),
    "history": bilstm_history,
    "test_metrics": bilstm_test_metrics
}

# GRU experiment record
gru_experiment = {
    "model": "GRU",
    "epochs": 5,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
    "trainable_parameters": sum(
        p.numel() for p in gru_model.parameters()
        if p.requires_grad
    ),
    "history": gru_history,
    "test_metrics": gru_test_metrics
}

# Save both records
with open(os.path.join(REPORT_DIR, "bilstm_experiment.json"), "w") as f:
    json.dump(bilstm_experiment, f, indent=4)

with open(os.path.join(REPORT_DIR, "gru_experiment.json"), "w") as f:
    json.dump(gru_experiment, f, indent=4)

print("BiLSTM and GRU experiment records saved successfully!")

BiLSTM and GRU experiment records saved successfully!


In [ ]:
import os
import json
import pandas as pd

# --------------------------------
# 1. Create model comparison table
# --------------------------------

comparison_data = [
    {
        "Model": "Simple RNN",
        "Validation_Accuracy": 0.5064,
        "Validation_Loss": 0.6991,
        "Test_Accuracy": None,
        "Test_Precision": None,
        "Test_Recall": None,
        "Test_F1": None,
        "Test_ROC_AUC": None
    },
    {
        "Model": "LSTM",
        "Validation_Accuracy": 0.7206,
        "Validation_Loss": 0.6097,
        "Test_Accuracy": 0.7056,
        "Test_Precision": 0.7596,
        "Test_Recall": 0.6016,
        "Test_F1": 0.6714,
        "Test_ROC_AUC": 0.7392
    },
    {
        "Model": "BiLSTM",
        "Validation_Accuracy": 0.8442,
        "Validation_Loss": 0.3755,
        "Test_Accuracy": bilstm_test_metrics["accuracy"],
        "Test_Precision": bilstm_test_metrics["precision"],
        "Test_Recall": bilstm_test_metrics["recall"],
        "Test_F1": bilstm_test_metrics["f1"],
        "Test_ROC_AUC": bilstm_test_metrics["roc_auc"]
    },
    {
        "Model": "GRU",
        "Validation_Accuracy": 0.8934,
        "Validation_Loss": 0.2592,
        "Test_Accuracy": gru_test_metrics["accuracy"],
        "Test_Precision": gru_test_metrics["precision"],
        "Test_Recall": gru_test_metrics["recall"],
        "Test_F1": gru_test_metrics["f1"],
        "Test_ROC_AUC": gru_test_metrics["roc_auc"]
    }
]

comparison_df = pd.DataFrame(comparison_data)

# Save CSV
COMPARISON_PATH = os.path.join(
    REPORT_DIR,
    "recurrent_model_comparison.csv"
)

comparison_df.to_csv(COMPARISON_PATH, index=False)

print("Comparison CSV created!")
print("\n", comparison_df.round(4))


# --------------------------------
# 2. Verify saved artifacts
# --------------------------------

expected_files = [
    os.path.join(MODEL_DIR, "lstm_model.pth"),
    os.path.join(MODEL_DIR, "bilstm_model.pth"),
    os.path.join(MODEL_DIR, "gru_epoch5_last.pth"),

    os.path.join(REPORT_DIR, "lstm_experiment.json"),
    os.path.join(REPORT_DIR, "bilstm_experiment.json"),
    os.path.join(REPORT_DIR, "gru_experiment.json"),
    COMPARISON_PATH
]

print("\nARTIFACT VERIFICATION")
print("=" * 55)

all_verified = True

for file_path in expected_files:

    exists = os.path.isfile(file_path)

    if exists:
        size_kb = os.path.getsize(file_path) / 1024
        print(f"[OK] {os.path.basename(file_path)} - {size_kb:.2f} KB")
    else:
        print(f"[MISSING] {file_path}")
        all_verified = False

print("=" * 55)

if all_verified:
    print("All expected artifacts verified successfully!")
else:
    print("Some artifacts are missing. Check the paths above.")


# --------------------------------
# 3. Verify CSV can be reloaded
# --------------------------------

verified_df = pd.read_csv(COMPARISON_PATH)

print("\nReloaded comparison CSV:")
display(verified_df)

Comparison CSV created!

         Model  Validation_Accuracy  Validation_Loss  Test_Accuracy  \
0  Simple RNN               0.5064           0.6991            NaN   
1        LSTM               0.7206           0.6097         0.7056   
2      BiLSTM               0.8442           0.3755         0.8464   
3         GRU               0.8934           0.2592         0.8884   

   Test_Precision  Test_Recall  Test_F1  Test_ROC_AUC  
0             NaN          NaN      NaN           NaN  
1          0.7596       0.6016   0.6714        0.7392  
2          0.8876       0.7932   0.8378        0.9230  
3          0.8903       0.8860   0.8881        0.9571  

ARTIFACT VERIFICATION
[OK] lstm_model.pth - 15519.37 KB
[OK] bilstm_model.pth - 16036.58 KB
[OK] gru_epoch5_last.pth - 15390.73 KB
[OK] lstm_experiment.json - 0.92 KB
[OK] bilstm_experiment.json - 1.03 KB
[OK] gru_experiment.json - 1.02 KB
[OK] recurrent_model_comparison.csv - 0.36 KB
All expected artifacts verified successfully!

Reloaded 

,Model,Validation_Accuracy,Validation_Loss,Test_Accuracy,Test_Precision,Test_Recall,Test_F1,Test_ROC_AUC
0,Simple RNN,0.5064,0.6991,NaN,NaN,NaN,NaN,NaN
1,LSTM,0.7206,0.6097,0.7056,0.759600,0.6016,0.671400,0.739200
2,BiLSTM,0.8442,0.3755,0.8464,0.887645,0.7932,0.837769,0.922967
3,GRU,0.8934,0.2592,0.8884,0.890273,0.8860,0.888132,0.957122
